In [ ]:
!pip install -U transformers

## Local Inference on GPU 
Model page: https://huggingface.co/google/gemma-4-E2B-it-qat-mobile-transformers

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/gemma-4-E2B-it-qat-mobile-transformers)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Load model directly
from transformers import AutoModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name_gemma="google/gemma-4-E2B-it-qat-mobile-transformers"
model_name="openbmb/MiniCPM5-1B"
model_openbmb="openbmb/MiniCPM5-1B"

In [ ]:

model = AutoModel.from_pretrained(model_name, dtype="auto").to(device)

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("openbmb/MiniCPM5-1B")
model = AutoModelForCausalLM.from_pretrained("openbmb/MiniCPM5-1B")


In [ ]:
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=120)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
# from transformers import AutoTokenizer, pipeline

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# pipeline = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     model_kwargs={"torch_dtype": "auto"},
# )

# # Example of generating chat/text
# messages = [
#     {"role": "user", "content": "What is your favorite color?"},
# ]

# prompt = pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# outputs = pipeline(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
# print(outputs[0]["generated_text"])

# Finetuning

In [ ]:
# Install necessary libraries for finetuning (if not already installed)
!pip install -q -U accelerate peft bitsandbytes transformers trl datasets

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

# --- 1. Load Model and Tokenizer for Finetuning with QLoRA ---
# The `model` variable from a previous cell is AutoModel, for finetuning
# we typically need AutoModelForCausalLM with BitsAndBytesConfig.
# Let's reload it for clarity in this finetuning example.

# model_id = "google/gemma-4-E2B-it-qat-mobile-transformers"
model_id = model_openbmb

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

# Load the model with 4-bit quantization
# Assuming 'model' from earlier cell is a suitable base for this if not reloaded
# If you've already loaded it as `AutoModel`, you might need to ensure it's `AutoModelForCausalLM`
# and can be quantizied. For a robust finetuning example, it's safer to load it here again.

tuned_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

tuned_model.config.use_cache = False
tuned_model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token


In [ ]:

# Prepare model for k-bit training
tuned_model = prepare_model_for_kbit_training(tuned_model)

# --- 2. Configure LoRA ---
lora_config = LoraConfig(
    r=16, # LoRA attention dimension
    lora_alpha=16, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Target all linear layers
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Only add bias to the LoRA layers
    task_type="CAUSAL_LM", # Task type for causal language modeling
)

# Do NOT call get_peft_model() here — SFTTrainer wraps the model when peft_config is passed.
# tuned_model = get_peft_model(tuned_model, lora_config)

# --- 3. Prepare a Sample Dataset ---
# For a real-world scenario, you would load your own dataset using `load_dataset`
# from the `datasets` library and format it appropriately.
# This is a simple dummy dataset for demonstration.

# Example instruction tuning dataset format
data = {
    "text": [
        "<start_of_turn>user\nWhat is the capital of France?<end_of_turn>\n<start_of_turn>model\nParis is the capital of France.<end_of_turn>",
        "<start_of_turn>user\nSuggest a healthy snack.\n<end_of_turn>\n<start_of_turn>model\nAlmonds or a piece of fruit like an apple are great healthy snack options.<end_of_turn>",
        "<start_of_turn>user\nExplain the concept of photosynthesis.\n<end_of_turn>\n<start_of_turn>model\nPhotosynthesis is the process by which green plants and some other organisms convert light energy into chemical energy.<end_of_turn>"
    ]
}

dataset = Dataset.from_dict(data)

# --- 4. Define Training Arguments ---
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma_finetuned", # Output directory for checkpoints and logs
    num_train_epochs=1, # Number of training epochs
    per_device_train_batch_size=2, # Batch size per GPU/CPU for training
    gradient_accumulation_steps=2, # Number of updates steps to accumulate before performing a backward/update pass
    optim="paged_adamw_8bit", # Optimizer to use
    save_steps=100, # Save checkpoint every X updates steps
    logging_steps=10, # Log every X updates steps
    learning_rate=2e-4, # Initial learning rate for AdamW optimizer
    weight_decay=0.001, # Weight decay for AdamW
    fp16=True, # Enable mixed precision training
    bf16=False, # Disable BF16 if using FP16
    max_grad_norm=0.3, # Max gradient norm
    max_steps=-1, # Don't limit training by steps, use epochs
    warmup_ratio=0.03, # Ratio of total steps for a linear warmup from 0 to learning_rate
    # group_by_length=True, # Group sequences of roughly the same length together to speed up training
    lr_scheduler_type="constant", # Learning rate scheduler type
    report_to="none" # Disable reporting to any tracking service
)

# --- 5. Initialize and Run SFTTrainer ---

trainer = SFTTrainer(
    model=tuned_model,          # plain (non-PEFT) base model
    train_dataset=dataset,
    peft_config=lora_config,    # SFTTrainer applies LoRA internally
    # dataset_text_field="text", # Name of the column containing the text data
    # tokenizer=tokenizer,
    args=training_args,
    # packing=False, # Whether to pack multiple short examples into one longer sequence to improve efficiency
    # max_seq_length=512, # Max sequence length to use for training
)

print("Starting finetuning...")
trainer.train()
print("Finetuning complete!")

# --- 6. (Optional) Save the finetuned model ---
# trainer.save_model("./gemma_finetuned_model")

# --- 7. (Optional) Merge LoRA adapters with the base model for inference ---
# from peft import AutoPeftModelForCausalLM
# merged_model = AutoPeftModelForCausalLM.from_pretrained(
#     "./gemma_finetuned_model",
#     device_map="auto",
#     torch_dtype=torch.bfloat16 # or torch.float16 depending on your hardware
# )
# merged_model.save_pretrained("gemma_merged_model", safe_serialization=True)
# tokenizer.save_pretrained("gemma_merged_model")


In [ ]:
# Duplicate cell removed — run the finetuning cell above.


In [ ]:
# Alternative (pick ONE approach, not both):
#
# Option A — let SFTTrainer apply LoRA (used in the cell above):
#   tuned_model = prepare_model_for_kbit_training(tuned_model)
#   trainer = SFTTrainer(model=tuned_model, peft_config=lora_config, ...)
#
# Option B — wrap manually, omit peft_config from SFTTrainer:
#   tuned_model = get_peft_model(tuned_model, lora_config)
#   trainer = SFTTrainer(model=tuned_model, ...)  # no peft_config

